# General Cross-Encoder Training (Model 2)

Trains a cross-corpus fallback expert for ambiguous queries where the router has low confidence.

## Step 1: Install Dependencies

In [ ]:
!pip install faiss-cpu

## Step 2: Train Baseline General CE (No Corpus Tokens)

Trains BGE-reranker-v2-m3 on all corpora combined **without** corpus tokens.

Uses ListNet loss, reads pre-built training groups from Stage 1, evaluates per-corpus NDCG@20.

In [1]:
# === Cell 2: Stage 2 — fine-tune the cross-encoder from saved groups (fp16 autocast, fp16 save)
import os, json, math, time, random
from pathlib import Path
from typing import List, Dict, Tuple, Union
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup


# ---------------------- Config (same knobs as original) ----------------------
OUTPUT_DIR     = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")
CE_MODEL_NAME  = os.getenv("CE_MODEL_NAME","BAAI/bge-reranker-v2-m3")

EPOCHS         = int(os.getenv("EPOCHS", "2"))
LR             = float(os.getenv("LR", "1e-5"))
BATCH_GROUPS   = int(os.getenv("BATCH_GROUPS", "2"))
GRAD_ACCUM     = int(os.getenv("GRAD_ACCUM", "1"))
MAX_LEN        = int(os.getenv("MAX_LEN", "384"))
WEIGHT_DECAY   = float(os.getenv("WEIGHT_DECAY", "0.02"))
CLIP_NORM      = float(os.getenv("CLIP_NORM", "1.0"))
TAU            = float(os.getenv("TAU", "0.9"))
VAL_EVAL_K     = int(os.getenv("VAL_EVAL_K", "20"))
SEED           = int(os.getenv("SEED", "42"))

# Optional toggles
LOSS_LABELED_ONLY  = int(os.getenv("LOSS_LABELED_ONLY", "0"))

# Eval speed controls
EVAL_BATCH_PAIRS   = int(os.getenv("EVAL_BATCH_PAIRS", "128"))
PAD_TO_MULTIPLE_OF = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))

# Paths to stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k70.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k70.jsonl"))

# Auto-match Stage-1 K_E5 if meta.json exists (unless paths explicitly overridden)
_meta_path = os.path.join(STAGE1_DIR, "meta.json")
if os.path.exists(_meta_path):
    try:
        _k_meta = json.load(open(_meta_path))["K_E5"]
        if "GROUPS_TRAIN_JSONL" not in os.environ:
            GROUPS_TRAIN_JSONL = os.path.join(STAGE1_DIR, f"groups_train_k{_k_meta}.jsonl")
        if "GROUPS_VAL_JSONL" not in os.environ:
            GROUPS_VAL_JSONL   = os.path.join(STAGE1_DIR, f"groups_val_k{_k_meta}.jsonl")
    except Exception:
        pass

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Allow TF32 for speed on Ampere (harmless with fp16 autocast)
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"[CONFIG] device={device}  EPOCHS={EPOCHS}  LR={LR}  BATCH_GROUPS={BATCH_GROUPS}x{GRAD_ACCUM}")
print(f"[EVAL]   EVAL_BATCH_PAIRS={EVAL_BATCH_PAIRS}  PAD_TO_MULTIPLE_OF={PAD_TO_MULTIPLE_OF}  MAX_LEN={MAX_LEN}")
print(f"[PATHS] OUT={OUTPUT_DIR}  TRAIN_GROUPS={GROUPS_TRAIN_JSONL}  VAL_GROUPS={GROUPS_VAL_JSONL}")

# ---------------------- Load groups ----------------------
def _read_jsonl(path):
    items=[]
    if not os.path.exists(path):
        return items
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

groups_train = _read_jsonl(GROUPS_TRAIN_JSONL)
groups_val   = _read_jsonl(GROUPS_VAL_JSONL)
print(f"[DATA] train groups={len(groups_train)}  val groups={len(groups_val)}")

# Skip groups with zero positives at dataset time (speeds training)
_before = len(groups_train)
groups_train = [g for g in groups_train if any(l > 0 for l in g["labels"])]
print(f"[DATA] filtered train groups with no positives: {_before - len(groups_train)} dropped; {len(groups_train)} remain")

# ---------------------- Datasets & Collate (listwise) ----------------------
class ListwiseDataset(torch.utils.data.Dataset):
    def __init__(self, groups): self.groups = groups
    def __len__(self): return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return g["query"], g["texts"], g["labels"], g["pids"]

tok_ce = AutoTokenizer.from_pretrained(CE_MODEL_NAME)

def collate_listwise(batch, tokenizer, max_len=512):
    # batch: list of (q, [texts], [labels], [pids])
    Q, P, gains, spans, PIDS = [], [], [], [], []
    cur = 0
    for q, texts, labs, pids in batch:
        Q += [q]*len(texts)
        P += texts
        gains += [float((2**int(l))-1) for l in labs]  # gains
        PIDS += pids
        spans.append((cur, cur+len(texts)))
        cur += len(texts)
    enc = tokenizer(Q, P, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return enc, torch.tensor(gains, dtype=torch.float32), spans, PIDS

def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    """
    ListNet loss over groups in `spans`.
    Compute softmax math in fp32 for stability even when using fp16 autocast.
    """
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0:
            continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

def listnet_loss_labeled_only(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        pos = g > 0
        if pos.sum() == 0:
            continue
        p_t = g[pos] / (g[pos].sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e][pos] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ---------------------- Model (FP32 weights) & Optim ----------------------
try:
    ce = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True, attn_implementation="sdpa"
    ).to(device)
except TypeError:
    ce = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True
    ).to(device)
ce = ce.float()

train_ds = ListwiseDataset(groups_train)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_GROUPS, shuffle=True,
    collate_fn=lambda b: collate_listwise(b, tok_ce, MAX_LEN),
    pin_memory=torch.cuda.is_available(), num_workers=2 if torch.cuda.is_available() else 0,
    persistent_workers=torch.cuda.is_available()
)

# -------- Pre-tokenize VAL once for FAST eval + keep per-corpus info --------
def build_val_pretok(groups, tokenizer, max_len=512, pad_multi=8):
    """
    Returns a dict:
      {
        'enc': {'input_ids': LongTensor[N,L], 'attention_mask': LongTensor[N,L], ...},
        'spans': List[Tuple[start,end]] per group,
        'pids': List[str] flattened,
        'rels': IntTensor[N] with 0..4 labels,
        'cases': List[str] of len(groups) with case_name per group (for per-corpus NDCG)
      }
    """
    all_input_ids, all_attn, all_ttids = [], [], []
    all_rels, all_pids, spans, cases = [], [], [], []
    cur = 0
    for g in groups:
        q, texts, labels, pids = g["query"], g["texts"], g["labels"], g["pids"]
        enc = tokenizer(
            [q] * len(texts),
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            pad_to_multiple_of=pad_multi,
            return_tensors="pt",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
        if "token_type_ids" in enc:
            all_ttids.append(enc["token_type_ids"])

        all_rels.extend([int(l) for l in labels])
        all_pids.extend(pids)
        spans.append((cur, cur + len(texts)))
        cases.append(g.get("case_name") or "unknown")
        cur += len(texts)

    if len(spans) == 0:
        # empty validation set
        return {"enc": {"input_ids": torch.empty(0, max_len, dtype=torch.long),
                        "attention_mask": torch.empty(0, max_len, dtype=torch.long)},
                "spans": [], "pids": [], "rels": torch.tensor([], dtype=torch.int16), "cases": []}

    input_ids = torch.cat(all_input_ids, dim=0)
    attention_mask = torch.cat(all_attn, dim=0)
    enc_full = {"input_ids": input_ids, "attention_mask": attention_mask}
    if len(all_ttids) > 0:
        enc_full["token_type_ids"] = torch.cat(all_ttids, dim=0)

    for k in list(enc_full.keys()):
        enc_full[k] = enc_full[k].pin_memory()

    rels = torch.tensor(all_rels, dtype=torch.int16)
    return {"enc": enc_full, "spans": spans, "pids": all_pids, "rels": rels, "cases": cases}

pretok_val = build_val_pretok(groups_val, tok_ce, max_len=MAX_LEN, pad_multi=PAD_TO_MULTIPLE_OF)
val_loader = pretok_val  # keep name for compatibility

# Optimizer / Scheduler
optimizer = torch.optim.AdamW(ce.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8, betas=(0.9, 0.999))
# -------- Simple poly scheduler: warmup 10%, final LR = 10% of base (never 0) --------
updates_per_epoch = math.ceil(len(train_loader) / max(1, GRAD_ACCUM))
num_update_steps  = max(1, updates_per_epoch * EPOCHS)

warmup_ratio = 0.10
warmup = max(1, int(warmup_ratio * num_update_steps))

# keep a small non-zero tail so last step isn't 0
lr_end_frac = 0.10                   # final LR = lr_end_frac * LR
lr_end = max(LR * lr_end_frac, 1e-8) # hard floor just in case

scheduler = get_polynomial_decay_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup,
    num_training_steps=num_update_steps,
    lr_end=lr_end,
    power=1.0
)

# AMP config: fp16 autocast + GradScaler on CUDA
use_amp = (device == "cuda")
amp_dtype = torch.float16 if use_amp else None
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# ---------------------- Eval helpers (NDCG@K) ----------------------
def _dcg_at_k(rels, k=20):
    return sum(((2**r-1)/math.log2(i+2) for i,r in enumerate(rels[:k])), 0.0)

def ndcg_at_k(ids_sorted: List[str], gt_map: Dict[str,int], k=20):
    rels = [gt_map.get(pid, 0) for pid in ids_sorted[:k]]
    dcg  = _dcg_at_k(rels, k)
    idcg = _dcg_at_k(sorted(gt_map.values(), reverse=True), k)
    return 0.0 if idcg==0.0 else float(dcg/idcg)

@torch.no_grad()
def evaluate_ndcg(model, loader_or_pretok: Union[dict, torch.utils.data.DataLoader], k=20) -> float:
    """
    Original overall NDCG (kept for compatibility).
    """
    overall, _ = evaluate_ndcg_breakdown(model, loader_or_pretok, k=k)
    return overall

@torch.no_grad()
def evaluate_ndcg_breakdown(model, loader_or_pretok: Union[dict, torch.utils.data.DataLoader], k=20):
    """
    Returns (overall_ndcg: float, per_case: Dict[str, {'ndcg': float, 'n': int}]).
    Fast path uses pre-tokenized batches and computes per-corpus NDCG via group spans + case names.
    """
    model.eval()

    # -------- Fast path (pre-tokenized) --------
    if isinstance(loader_or_pretok, dict) and "enc" in loader_or_pretok:
        enc_full = loader_or_pretok["enc"]
        spans    = loader_or_pretok["spans"]
        rels_all = loader_or_pretok["rels"]
        cases    = loader_or_pretok.get("cases", ["unknown"]*len(spans))

        if len(spans) == 0:
            return 0.0, {}

        N = enc_full["input_ids"].size(0)
        scores = torch.empty(N, dtype=torch.float32)

        start = 0
        while start < N:
            end = min(start + EVAL_BATCH_PAIRS, N)
            sl = slice(start, end)
            inputs = {k: v[sl].to(device, non_blocking=True) for k, v in enc_full.items()}

            ctx = (torch.autocast(device_type="cuda", dtype=torch.float16)
                   if device=="cuda" else torch.cpu.amp.autocast(enabled=False))
            with torch.inference_mode(), ctx:
                logits = model(**inputs).logits
                if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
                elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
                else: s = logits.view(-1).float()

            scores[sl] = s.detach().cpu()
            start = end

        denom = 1.0 / np.log2(np.arange(2, k + 2))
        s_np = scores.numpy()
        r_np = rels_all.numpy()

        ndcgs = []
        per_case_sum = {}
        per_case_cnt = {}

        for gi, (st, ed) in enumerate(spans):
            group_scores = s_np[st:ed]
            group_rels   = r_np[st:ed]

            order = np.argsort(-group_scores)
            rel_sorted = group_rels[order][:k]
            dcg = ((np.power(2.0, rel_sorted, dtype=np.float64) - 1.0) * denom[:len(rel_sorted)]).sum()

            ideal = np.sort(group_rels)[::-1][:k]
            idcg = ((np.power(2.0, ideal, dtype=np.float64) - 1.0) * denom[:len(ideal)]).sum()

            nd = 0.0 if idcg <= 0.0 else float(dcg / idcg)
            ndcgs.append(nd)

            c = cases[gi]
            per_case_sum[c] = per_case_sum.get(c, 0.0) + nd
            per_case_cnt[c] = per_case_cnt.get(c, 0) + 1

        overall = float(np.mean(ndcgs)) if ndcgs else 0.0
        per_case = {c: {"ndcg": (per_case_sum[c] / max(1, per_case_cnt[c])), "n": per_case_cnt[c]}
                    for c in per_case_sum.keys()}
        return overall, per_case

    # -------- Fallback: slow path (rarely used here) --------
    ndcgs=[]
    per_case_sum, per_case_cnt = {}, {}
    for enc_cpu, gains_cpu, spans, pids in loader_or_pretok:
        enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        logits = model(**enc).logits
        if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
        elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
        else: s = logits.view(-1).float()
        scores = s.detach().cpu().numpy().tolist()
        # NOTE: slow path cannot recover corpus names, so everything becomes 'all'
        for (st,ed) in spans:
            group_pids = pids[st:ed]
            group_scores = scores[st:ed]
            g_local = gains_cpu[st:ed].tolist()
            rels_local = []
            for g in g_local:
                if g <= 0: rels_local.append(0)
                elif math.isclose(g,1.0): rels_local.append(1)
                elif math.isclose(g,3.0): rels_local.append(2)
                elif math.isclose(g,7.0): rels_local.append(3)
                elif math.isclose(g,15.0): rels_local.append(4)
                else:
                    rels_local.append(int(round(math.log2(g+1))))
            gt = {pid: rel for pid, rel in zip(group_pids, rels_local)}
            ranked = [pid for pid,_ in sorted(zip(group_pids, group_scores), key=lambda x:x[1], reverse=True)]
            nd = ndcg_at_k(ranked, gt, k=k)
            ndcgs.append(nd)
            per_case_sum["all"] = per_case_sum.get("all", 0.0) + nd
            per_case_cnt["all"] = per_case_cnt.get("all", 0) + 1
    overall = float(np.mean(ndcgs)) if ndcgs else 0.0
    per_case = {c: {"ndcg": (per_case_sum[c] / max(1, per_case_cnt[c])), "n": per_case_cnt[c]}
                for c in per_case_sum.keys()}
    return overall, per_case

def _print_per_corpus(tag: str, per_case: Dict[str, Dict[str, Union[float,int]]], sort_keys=True):
    if not per_case:
        print(f"[{tag}] No validation groups available.")
        return
    items = list(per_case.items())
    if sort_keys:
        items.sort(key=lambda kv: kv[0])
    for c, m in items:
        print(f"[{tag}] {c:>12}  NDCG@{VAL_EVAL_K}={m['ndcg']:.4f}  (n={m['n']})")


# ---------------------- Helpers: model size on disk ----------------------
def _dir_size_bytes(path: str) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try: total += os.path.getsize(fp)
            except OSError: pass
    return total

def _human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB"]
    i = 0; v = float(n)
    while v >= 1024.0 and i < len(units)-1: v /= 1024.0; i += 1
    return f"{v:.2f} {units[i]}"

def _save_fp16_and_report(model, tokenizer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    orig_dtype = next(model.parameters()).dtype
    try:
        model.to(dtype=torch.float16)  # cast to fp16 just for saving (smaller)
        model.save_pretrained(out_dir, safe_serialization=True)  # safetensors
        tokenizer.save_pretrained(out_dir)
    finally:
        model.to(dtype=orig_dtype)  # restore original dtype (fp32) for continued training
    sz = _dir_size_bytes(out_dir)
    print(f"[SAVE] Checkpoint saved (fp16) → {out_dir}  |  folder size = {_human_bytes(sz)}")

# ---------------------- Baseline before training ----------------------
print("\n[VAL] evaluating pretrained CE on E5@K sets …")
baseline_overall, baseline_cases = evaluate_ndcg_breakdown(ce, val_loader, k=VAL_EVAL_K)
print(f"[VAL] Baseline NDCG@{VAL_EVAL_K}: {baseline_overall:.4f}")
_print_per_corpus("VAL-BASE", baseline_cases)

# ---------------------- Train (fp16 autocast, fp32 weights) ----------------------
best = baseline_overall
compute_loss = (listnet_loss_labeled_only if LOSS_LABELED_ONLY else listnet_loss)

for ep in range(1, EPOCHS+1):
    ce.train()
    running, micro = 0.0, 0
    t0 = time.perf_counter()

    for step, (enc_cpu, gains_cpu, spans, _pids) in enumerate(train_loader, 1):
        enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        gains = gains_cpu.to(device, non_blocking=True)

        if use_amp:
            ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            class _NoOp:
                def __enter__(self): pass
                def __exit__(self, *args): return False
            ctx = _NoOp()

        with ctx:
            logits = ce(**enc).logits
            if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
            else: s = logits.view(-1).float()
            loss = compute_loss(s, gains, spans, tau=TAU)

        micro += 1
        if scaler is not None and use_amp:
            scaler.scale(loss).backward()
            if CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ce.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                scaler.step(optimizer); scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        else:
            loss.backward()
            if CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(ce.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        running += float(loss.item())

        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.perf_counter() - t0
            print(f"[train e{ep}/{EPOCHS}] step {step}/{len(train_loader)} "
                  f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"steps/s={step/max(1e-6,elapsed):.2f}")

    # Validate (FAST path uses pretokenized val via evaluate_ndcg_breakdown)
    val_overall, val_cases = evaluate_ndcg_breakdown(ce, val_loader, k=VAL_EVAL_K)
    print(f"[VAL] epoch {ep}  NDCG@{VAL_EVAL_K}={val_overall:.4f}  (baseline {baseline_overall:.4f})")
    _print_per_corpus(f"VAL-e{ep}", val_cases)

    if val_overall > best + 1e-4:
        best = val_overall
        _save_fp16_and_report(ce, tok_ce, OUTPUT_DIR)

print(f"\n[RESULT] Best VAL NDCG@{VAL_EVAL_K}: {best:.4f} (baseline {baseline_overall:.4f})")
print(f"[DONE] Saved fp16 model to: {OUTPUT_DIR}")


[CONFIG] device=cuda  EPOCHS=2  LR=1e-05  BATCH_GROUPS=2x1
[EVAL]   EVAL_BATCH_PAIRS=128  PAD_TO_MULTIPLE_OF=8  MAX_LEN=384
[PATHS] OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16  TRAIN_GROUPS=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1/groups_train_k70.jsonl  VAL_GROUPS=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1/groups_val_k70.jsonl
[DATA] train groups=1728  val groups=306
[DATA] filtered train groups with no positives: 44 dropped; 1684 remain


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



[VAL] evaluating pretrained CE on E5@K sets …
[VAL] Baseline NDCG@20: 0.5942
[VAL-BASE] mafat_retrieval_knesset_corpus  NDCG@20=0.4725  (n=68)
[VAL-BASE] mafat_retrieval_kz_corpus  NDCG@20=0.5157  (n=122)
[VAL-BASE] mafat_retrieval_wikipedia_corpus  NDCG@20=0.7482  (n=116)
[train e1/2] step 50/842 loss(avg)=3.6281  lr=2.98e-06 steps/s=1.11
[train e1/2] step 100/842 loss(avg)=3.4747  lr=5.95e-06 steps/s=1.12
[train e1/2] step 150/842 loss(avg)=3.3094  lr=8.93e-06 steps/s=1.12
[train e1/2] step 200/842 loss(avg)=3.2591  lr=9.81e-06 steps/s=1.12
[train e1/2] step 250/842 loss(avg)=3.2530  lr=9.51e-06 steps/s=1.12
[train e1/2] step 300/842 loss(avg)=3.2180  lr=9.22e-06 steps/s=1.12
[train e1/2] step 350/842 loss(avg)=3.1881  lr=8.92e-06 steps/s=1.12
[train e1/2] step 400/842 loss(avg)=3.1589  lr=8.62e-06 steps/s=1.12
[train e1/2] step 450/842 loss(avg)=3.1539  lr=8.33e-06 steps/s=1.12
[train e1/2] step 500/842 loss(avg)=3.1305  lr=8.03e-06 steps/s=1.12
[train e1/2] step 550/842 loss(avg)=

## Step 3: Train General Expert with Corpus Tokens

Trains BGE-reranker-v2-m3 on **all corpora combined** with corpus-identifying tokens.

**Key difference from Step 2**: Prepends special tokens (`<CORP:WIKIPEDIA>`, `<CORP:KZ>`, `<CORP:KNESSET>`) to queries, allowing the model to learn corpus-specific patterns while sharing parameters.

This is the **General Expert** used as fallback when the router has low confidence.

In [2]:
# === Cell 2: Stage 2 — fine-tune the cross-encoder with per-corpus token (fp16 autocast, fp16 save) ===
import os, json, math, time, random, re
from pathlib import Path
from typing import List, Dict, Tuple, Union
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_polynomial_decay_schedule_with_warmup

# ---------------------- Config ----------------------
OUTPUT_DIR     = os.getenv("OUTPUT_DIR",   "/content/ce_ft/bge-reranker-v2-m3_t50_fp16")
CE_MODEL_NAME  = os.getenv("CE_MODEL_NAME","BAAI/bge-reranker-v2-m3")

EPOCHS         = int(os.getenv("EPOCHS", "1"))
LR             = float(os.getenv("LR", "1e-5"))
BATCH_GROUPS   = int(os.getenv("BATCH_GROUPS", "2"))
GRAD_ACCUM     = int(os.getenv("GRAD_ACCUM", "1"))
MAX_LEN        = int(os.getenv("MAX_LEN", "384"))
WEIGHT_DECAY   = float(os.getenv("WEIGHT_DECAY", "0.02"))
CLIP_NORM      = float(os.getenv("CLIP_NORM", "1.0"))
TAU            = float(os.getenv("TAU", "0.9"))
VAL_EVAL_K     = int(os.getenv("VAL_EVAL_K", "20"))
SEED           = int(os.getenv("SEED", "42"))

# Optional toggles
LOSS_LABELED_ONLY   = int(os.getenv("LOSS_LABELED_ONLY", "0"))

# >>> NEW: corpus-token controls <<<
ENABLE_CORPUS_TOKEN = int(os.getenv("ENABLE_CORPUS_TOKEN", "1"))
CORPUS_TOKEN_POS    = os.getenv("CORPUS_TOKEN_POS", "prefix_query")  # prefix_query | prefix_passage | prefix_both

# Eval speed controls
EVAL_BATCH_PAIRS   = int(os.getenv("EVAL_BATCH_PAIRS", "128"))
PAD_TO_MULTIPLE_OF = int(os.getenv("PAD_TO_MULTIPLE_OF", "8"))

# Paths to stage-1 outputs
STAGE1_DIR = os.getenv("STAGE1_DIR", os.path.join(OUTPUT_DIR, "stage1"))
GROUPS_TRAIN_JSONL = os.getenv("GROUPS_TRAIN_JSONL", os.path.join(STAGE1_DIR, f"groups_train_k70.jsonl"))
GROUPS_VAL_JSONL   = os.getenv("GROUPS_VAL_JSONL",   os.path.join(STAGE1_DIR, f"groups_val_k70.jsonl"))

# Auto-match Stage-1 K_E5 if meta.json exists (unless paths explicitly overridden)
_meta_path = os.path.join(STAGE1_DIR, "meta.json")
if os.path.exists(_meta_path):
    try:
        _k_meta = json.load(open(_meta_path))["K_E5"]
        if "GROUPS_TRAIN_JSONL" not in os.environ:
            GROUPS_TRAIN_JSONL = os.path.join(STAGE1_DIR, f"groups_train_k{_k_meta}.jsonl")
        if "GROUPS_VAL_JSONL" not in os.environ:
            GROUPS_VAL_JSONL   = os.path.join(STAGE1_DIR, f"groups_val_k{_k_meta}.jsonl")
    except Exception:
        pass

# ---------------------- Setup ----------------------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"[CONFIG] device={device}  EPOCHS={EPOCHS}  LR={LR}  BATCH_GROUPS={BATCH_GROUPS}x{GRAD_ACCUM}")
print(f"[EVAL]   EVAL_BATCH_PAIRS={EVAL_BATCH_PAIRS}  PAD_TO_MULTIPLE_OF={PAD_TO_MULTIPLE_OF}  MAX_LEN={MAX_LEN}")
print(f"[PATHS] OUT={OUTPUT_DIR}  TRAIN_GROUPS={GROUPS_TRAIN_JSONL}  VAL_GROUPS={GROUPS_VAL_JSONL}")
print(f"[CORPUS TOKENS] enabled={bool(ENABLE_CORPUS_TOKEN)}  position={CORPUS_TOKEN_POS}")

# ---------------------- Load groups ----------------------
def _read_jsonl(path):
    items=[]
    if not os.path.exists(path):
        return items
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

groups_train = _read_jsonl(GROUPS_TRAIN_JSONL)
groups_val   = _read_jsonl(GROUPS_VAL_JSONL)
print(f"[DATA] train groups={len(groups_train)}  val groups={len(groups_val)}")

# Skip groups with zero positives (speeds training)
_before = len(groups_train)
groups_train = [g for g in groups_train if any(l > 0 for l in g["labels"])]
print(f"[DATA] filtered train groups with no positives: {_before - len(groups_train)} dropped; {len(groups_train)} remain")

# ---------------------- Corpus token setup ----------------------
def _norm_case(x: str) -> str:
    s = (x or "").strip().lower()
    if not s: return "unknown"
    # normalize common variants
    if "wiki" in s: return "wikipedia"
    if "knesset" in s: return "knesset"
    if s in {"kz", "kol zchut", "kol-zchut", "kolzchut"} or "kol" in s and "zchut" in s: return "kz"
    return s

ALL_CASES = sorted({_norm_case(g.get("case_name")) for g in (groups_train + groups_val)})
if "unknown" not in ALL_CASES:
    ALL_CASES.append("unknown")

CASE2TOKEN = {c: f"<CORP:{c.upper()}>" for c in ALL_CASES}
SPECIAL_CASE_TOKENS = list(CASE2TOKEN.values())

# ---------------------- Datasets & Collate (listwise) ----------------------
tok_ce = AutoTokenizer.from_pretrained(CE_MODEL_NAME)

if ENABLE_CORPUS_TOKEN and len(SPECIAL_CASE_TOKENS) > 0:
    tok_ce.add_special_tokens({"additional_special_tokens": SPECIAL_CASE_TOKENS})
    print(f"[CORPUS TOKENS] added to tokenizer: {SPECIAL_CASE_TOKENS}")

def _decorate_pair(q: str, p: str, case_name: str) -> Tuple[str, str]:
    if not ENABLE_CORPUS_TOKEN:
        return q, p
    tok = CASE2TOKEN.get(_norm_case(case_name), CASE2TOKEN.get("unknown", "<CORP:UNKNOWN>"))
    if CORPUS_TOKEN_POS == "prefix_passage":
        return q, f"{tok} {p}"
    elif CORPUS_TOKEN_POS == "prefix_both":
        return f"{tok} {q}", f"{tok} {p}"
    else:  # "prefix_query"
        return f"{tok} {q}", p

class ListwiseDataset(torch.utils.data.Dataset):
    def __init__(self, groups): self.groups = groups
    def __len__(self): return len(self.groups)
    def __getitem__(self, i):
        g = self.groups[i]
        return g["query"], g["texts"], g["labels"], g["pids"], (g.get("case_name") or "unknown")

def collate_listwise(batch, tokenizer, max_len=512):
    # batch: list of (q, [texts], [labels], [pids], case_name)
    Q, P, gains, spans, PIDS = [], [], [], [], []
    cur = 0
    for q, texts, labs, pids, case_name in batch:
        # apply corpus tokening consistently
        for t in texts:
            q2, p2 = _decorate_pair(q, t, case_name)
            Q.append(q2); P.append(p2)
        gains += [float((2**int(l))-1) for l in labs]
        PIDS += pids
        spans.append((cur, cur+len(texts)))
        cur += len(texts)
    enc = tokenizer(Q, P, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    return enc, torch.tensor(gains, dtype=torch.float32), spans, PIDS

def listnet_loss(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        if torch.count_nonzero(g) == 0: continue
        p_t = g / (g.sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

def listnet_loss_labeled_only(scores: torch.Tensor, gains: torch.Tensor, spans, tau=1.0):
    scores = scores.float(); gains = gains.float()
    loss_terms = []
    for s, e in spans:
        g = gains[s:e]
        pos = g > 0
        if pos.sum() == 0: continue
        p_t = g[pos] / (g[pos].sum() + 1e-12)
        log_p_s = torch.log_softmax(scores[s:e][pos] / tau, dim=0)
        loss_terms.append(-(p_t * log_p_s).sum())
    return torch.stack(loss_terms).mean() if loss_terms else scores.sum() * 0.0

# ---------------------- Model & Optim ----------------------
try:
    ce = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True, attn_implementation="sdpa"
    ).to(device)
except TypeError:
    ce = AutoModelForSequenceClassification.from_pretrained(
        CE_MODEL_NAME, trust_remote_code=True
    ).to(device)
ce = ce.float()

# if we added tokens, grow embeddings
if ENABLE_CORPUS_TOKEN and len(SPECIAL_CASE_TOKENS) > 0:
    ce.resize_token_embeddings(len(tok_ce))
    print(f"[CORPUS TOKENS] resized model embeddings → {len(tok_ce)} vocab size")

train_ds = ListwiseDataset(groups_train)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_GROUPS, shuffle=True,
    collate_fn=lambda b: collate_listwise(b, tok_ce, MAX_LEN),
    pin_memory=torch.cuda.is_available(), num_workers=2 if torch.cuda.is_available() else 0,
    persistent_workers=torch.cuda.is_available()
)

# -------- Pre-tokenize VAL once (with corpus tokens) --------
def build_val_pretok(groups, tokenizer, max_len=512, pad_multi=8):
    """
    Returns:
      {
        'enc': {'input_ids': LongTensor[N,L], 'attention_mask': LongTensor[N,L], ...},
        'spans': [(start,end)] per group,
        'pids': List[str],
        'rels': IntTensor[N] (0..4),
        'cases': List[str] per group
      }
    """
    if len(groups) == 0:
        return {"enc": {"input_ids": torch.empty(0, max_len, dtype=torch.long),
                        "attention_mask": torch.empty(0, max_len, dtype=torch.long)},
                "spans": [], "pids": [], "rels": torch.tensor([], dtype=torch.int16), "cases": []}

    all_input_ids, all_attn, all_ttids = [], [], []
    all_rels, all_pids, spans, cases = [], [], [], []
    cur = 0
    for g in groups:
        q, texts, labels, pids = g["query"], g["texts"], g["labels"], g["pids"]
        case_name = g.get("case_name") or "unknown"
        Q2, P2 = [], []
        for t in texts:
            q2, p2 = _decorate_pair(q, t, case_name)
            Q2.append(q2); P2.append(p2)
        enc = tokenizer(
            Q2, P2,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            pad_to_multiple_of=pad_multi,
            return_tensors="pt",
        )
        all_input_ids.append(enc["input_ids"])
        all_attn.append(enc["attention_mask"])
        if "token_type_ids" in enc:
            all_ttids.append(enc["token_type_ids"])

        all_rels.extend([int(l) for l in labels])
        all_pids.extend(pids)
        spans.append((cur, cur + len(texts)))
        cases.append(_norm_case(case_name))
        cur += len(texts)

    input_ids = torch.cat(all_input_ids, dim=0)
    attention_mask = torch.cat(all_attn, dim=0)
    enc_full = {"input_ids": input_ids, "attention_mask": attention_mask}
    if len(all_ttids) > 0:
        enc_full["token_type_ids"] = torch.cat(all_ttids, dim=0)
    for k in list(enc_full.keys()):
        enc_full[k] = enc_full[k].pin_memory()
    rels = torch.tensor(all_rels, dtype=torch.int16)
    return {"enc": enc_full, "spans": spans, "pids": all_pids, "rels": rels, "cases": cases}

pretok_val = build_val_pretok(groups_val, tok_ce, max_len=MAX_LEN, pad_multi=PAD_TO_MULTIPLE_OF)
val_loader = pretok_val

# Optimizer / Scheduler
optimizer = torch.optim.AdamW(ce.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, eps=1e-8, betas=(0.9, 0.999))
updates_per_epoch = max(1, math.ceil(len(train_loader) / max(1, GRAD_ACCUM)))
num_update_steps  = max(1, updates_per_epoch * EPOCHS)
warmup_ratio = 0.10
warmup = max(1, int(warmup_ratio * num_update_steps))

lr_end_frac = 0.10
lr_end = max(LR * lr_end_frac, 1e-8)

scheduler = get_polynomial_decay_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup,
    num_training_steps=num_update_steps,
    lr_end=lr_end,
    power=1.0
)

# AMP
use_amp = (device == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# ---------------------- Eval (overall + per-corpus NDCG@K) ----------------------
def _dcg_at_k(rels, k=20):
    return sum(((2**r-1)/math.log2(i+2) for i,r in enumerate(rels[:k])), 0.0)

@torch.no_grad()
def evaluate_ndcg_breakdown(model, loader_or_pretok: Union[dict, torch.utils.data.DataLoader], k=20):
    model.eval()
    if not isinstance(loader_or_pretok, dict) or "enc" not in loader_or_pretok or len(loader_or_pretok["spans"]) == 0:
        return 0.0, {}
    enc_full = loader_or_pretok["enc"]
    spans    = loader_or_pretok["spans"]
    rels_all = loader_or_pretok["rels"]
    cases    = loader_or_pretok.get("cases", ["unknown"]*len(spans))

    N = enc_full["input_ids"].size(0)
    scores = torch.empty(N, dtype=torch.float32)

    start = 0
    while start < N:
        end = min(start + EVAL_BATCH_PAIRS, N)
        sl = slice(start, end)
        inputs = {k: v[sl].to(device, non_blocking=True) for k, v in enc_full.items()}
        ctx = (torch.autocast(device_type="cuda", dtype=torch.float16)
               if device=="cuda" else torch.cpu.amp.autocast(enabled=False))
        with torch.inference_mode(), ctx:
            logits = model(**inputs).logits
            if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
            else: s = logits.view(-1).float()
        scores[sl] = s.detach().cpu()
        start = end

    denom = 1.0 / np.log2(np.arange(2, k + 2))
    s_np = scores.numpy()
    r_np = rels_all.numpy()

    ndcgs = []
    per_case_sum, per_case_cnt = {}, {}
    for gi, (st, ed) in enumerate(spans):
        group_scores = s_np[st:ed]
        group_rels   = r_np[st:ed]
        order = np.argsort(-group_scores)
        rel_sorted = group_rels[order][:k]
        dcg = ((np.power(2.0, rel_sorted, dtype=np.float64) - 1.0) * denom[:len(rel_sorted)]).sum()
        ideal = np.sort(group_rels)[::-1][:k]
        idcg = ((np.power(2.0, ideal, dtype=np.float64) - 1.0) * denom[:len(ideal)]).sum()
        nd = 0.0 if idcg <= 0.0 else float(dcg / idcg)
        ndcgs.append(nd)
        c = cases[gi]
        per_case_sum[c] = per_case_sum.get(c, 0.0) + nd
        per_case_cnt[c] = per_case_cnt.get(c, 0) + 1

    overall = float(np.mean(ndcgs)) if ndcgs else 0.0
    per_case = {c: {"ndcg": (per_case_sum[c] / max(1, per_case_cnt[c])), "n": per_case_cnt[c]} for c in per_case_sum}
    return overall, per_case

def _print_per_corpus(tag: str, per_case: Dict[str, Dict[str, Union[float,int]]], sort_keys=True):
    if not per_case:
        print(f"[{tag}] No validation groups available.")
        return
    items = list(per_case.items())
    if sort_keys:
        items.sort(key=lambda kv: kv[0])
    for c, m in items:
        print(f"[{tag}] {c:>12}  NDCG@{VAL_EVAL_K}={m['ndcg']:.4f}  (n={m['n']})")

# ---------------------- Baseline before training ----------------------
print("\n[VAL] evaluating pretrained CE on E5@K sets …")
baseline_overall, baseline_cases = evaluate_ndcg_breakdown(ce, val_loader, k=VAL_EVAL_K)
print(f"[VAL] Baseline NDCG@{VAL_EVAL_K}: {baseline_overall:.4f}")
_print_per_corpus("VAL-BASE", baseline_cases)

# ---------------------- Train ----------------------
best = baseline_overall
compute_loss = (listnet_loss_labeled_only if LOSS_LABELED_ONLY else listnet_loss)

for ep in range(1, EPOCHS+1):
    ce.train()
    running, micro = 0.0, 0
    t0 = time.perf_counter()

    for step, (enc_cpu, gains_cpu, spans, _pids) in enumerate(train_loader, 1):
        enc = {k: v.to(device, non_blocking=True) for k,v in enc_cpu.items()}
        gains = gains_cpu.to(device, non_blocking=True)

        if device == "cuda":
            ctx = torch.autocast(device_type="cuda", dtype=torch.float16)
        else:
            class _NoOp:
                def __enter__(self): pass
                def __exit__(self, *args): return False
            ctx = _NoOp()

        with ctx:
            logits = ce(**enc).logits
            if logits.dim()==2 and logits.shape[1]==1: s = logits.squeeze(-1).float()
            elif logits.dim()==2 and logits.shape[1]==2: s = logits[:,1].float()
            else: s = logits.view(-1).float()
            loss = compute_loss(s, gains, spans, tau=TAU)

        micro += 1
        if device == "cuda":
            scaler.scale(loss).backward()
            if CLIP_NORM > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ce.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                scaler.step(optimizer); scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        else:
            loss.backward()
            if CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(ce.parameters(), CLIP_NORM)
            if step % GRAD_ACCUM == 0:
                optimizer.step(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        running += float(loss.item())
        if step % 50 == 0 or step == len(train_loader):
            elapsed = time.perf_counter() - t0
            print(f"[train e{ep}/{EPOCHS}] step {step}/{len(train_loader)} "
                  f"loss(avg)={running/micro:.4f}  lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"steps/s={step/max(1e-6,elapsed):.2f}")

    # Validate
    val_overall, val_cases = evaluate_ndcg_breakdown(ce, val_loader, k=VAL_EVAL_K)
    print(f"[VAL] epoch {ep}  NDCG@{VAL_EVAL_K}={val_overall:.4f}  (baseline {baseline_overall:.4f})")
    _print_per_corpus(f"VAL-e{ep}", val_cases)

    if val_overall > best + 1e-4:
        best = val_overall
        # save compact fp16
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        orig_dtype = next(ce.parameters()).dtype
        try:
            ce.to(dtype=torch.float16)
            ce.save_pretrained(OUTPUT_DIR, safe_serialization=True)
            tok_ce.save_pretrained(OUTPUT_DIR)
        finally:
            ce.to(dtype=orig_dtype)
        # simple size report
        def _dir_size_bytes(path: str) -> int:
            tot=0
            for r,_,fs in os.walk(path):
                for f in fs:
                    try: tot += os.path.getsize(os.path.join(r,f))
                    except OSError: pass
            return tot
        sz = _dir_size_bytes(OUTPUT_DIR)
        units = ["B","KB","MB","GB","TB"]; i=0; v=float(sz)
        while v>=1024. and i<len(units)-1: v/=1024.; i+=1
        print(f"[SAVE] Checkpoint saved (fp16) → {OUTPUT_DIR}  |  folder size = {v:.2f} {units[i]}")

print(f"\n[RESULT] Best VAL NDCG@{VAL_EVAL_K}: {best:.4f} (baseline {baseline_overall:.4f})")
print(f"[DONE] Saved fp16 model to: {OUTPUT_DIR}")


[CONFIG] device=cuda  EPOCHS=1  LR=1e-05  BATCH_GROUPS=2x1
[EVAL]   EVAL_BATCH_PAIRS=128  PAD_TO_MULTIPLE_OF=8  MAX_LEN=384
[PATHS] OUT=/content/ce_ft/bge-reranker-v2-m3_t50_fp16  TRAIN_GROUPS=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1/groups_train_k70.jsonl  VAL_GROUPS=/content/ce_ft/bge-reranker-v2-m3_t50_fp16/stage1/groups_val_k70.jsonl
[CORPUS TOKENS] enabled=True  position=prefix_query
[DATA] train groups=1728  val groups=306
[DATA] filtered train groups with no positives: 44 dropped; 1684 remain
[CORPUS TOKENS] added to tokenizer: ['<CORP:KNESSET>', '<CORP:MAFAT_RETRIEVAL_KZ_CORPUS>', '<CORP:WIKIPEDIA>', '<CORP:UNKNOWN>']
[CORPUS TOKENS] resized model embeddings → 250006 vocab size

[VAL] evaluating pretrained CE on E5@K sets …
[VAL] Baseline NDCG@20: 0.5960
[VAL-BASE]      knesset  NDCG@20=0.4678  (n=68)
[VAL-BASE] mafat_retrieval_kz_corpus  NDCG@20=0.5179  (n=122)
[VAL-BASE]    wikipedia  NDCG@20=0.7532  (n=116)
[train e1/1] step 50/842 loss(avg)=3.4969  lr=5.95e-06 step

In [5]:
!zip -r -q "/content/bge-reranker-v2-m3_t50_fp16_general_0.6228.zip" "/content/ce_ft/bge-reranker-v2-m3_t50_fp16_general_0.6228"


## Step 4: Export Model

Zip and copy trained model to Google Drive for persistence.

In [6]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

# === Configure ===
SRC_ZIP      = "/content/bge-reranker-v2-m3_t50_fp16_general_0.6228.zip"     # <-- your zip
DEST_DIR     = "/content/drive/MyDrive/mafat_models"      # <-- Drive folder
OVERWRITE    = False                                      # set True to overwrite if exists
AUTO_RENAME  = True                                       # add timestamp if file exists and not overwriting

# === Ensure destination exists ===
os.makedirs(DEST_DIR, exist_ok=True)

# === Decide destination filename ===
base_name = os.path.basename(SRC_ZIP)
dest_path = os.path.join(DEST_DIR, base_name)

if os.path.exists(dest_path):
    if OVERWRITE:
        print(f"[INFO] Overwriting existing file at: {dest_path}")
    elif AUTO_RENAME:
        ts = time.strftime("%Y%m%d_%H%M%S")
        name, ext = os.path.splitext(base_name)
        dest_path = os.path.join(DEST_DIR, f"{name}_{ts}{ext}")
        print(f"[INFO] File exists. Auto-renaming to: {dest_path}")
    else:
        raise FileExistsError(f"Destination already exists: {dest_path}. "
                              f"Set OVERWRITE=True or AUTO_RENAME=True.")

# === Copy with metadata ===
shutil.copy2(SRC_ZIP, dest_path)
print(f"[DONE] Copied to Google Drive:\n{dest_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[DONE] Copied to Google Drive:
/content/drive/MyDrive/mafat_models/bge-reranker-v2-m3_t50_fp16_general_0.6228.zip
